# Extract Reusable EIGER Detector Corrections

## Purpose and Scope

Use a dense calibration scan to estimate detector-fixed pixel and eight-sensor multiplicative response corrections, then save a calibration HDF5 for later scans. This notebook starts from raw counts, a supplied goniometer JSON, this scan's measured I0, and native detector metadata. It does not use earlier correction maps or smooth the measured scattering pattern.

The supported detector is the documented EIGER2 CdTe 500K layout, shape 512 x 1028 with two sensor rows and four sensor columns. An arbitrary **compatible** scan can be selected; changing detector layout or scattering-plane convention requires changing and revalidating the implementation, not just the filename. A scan must have enough angular overlap and repeated observations of pixels. The code explicitly fails if a sensor lacks sufficient support.

## How To Use

1. Keep `eiger_workflow.py` in the same folder as both notebooks. Select a Python kernel containing NumPy, h5py, matplotlib, pyFAI, and xraylib; run the interpreter-check cell.
2. Edit `DATA_DIRECTORY`, `SCAN_PATH`, and `GEOMETRY_PATH` in the settings cell. Choose a fresh `CALIBRATION_OUTPUT` for every experiment; existing files are never overwritten.
3. By default the companion files are derived from the NeXus stem: `-fastcs_eiger_000001.h5`, `-i0.h5`, and `-fastcs_eiger_meta.h5`. Set `DETECTOR_PATH`, `MONITOR_PATH`, or `METADATA_PATH` explicitly for different locations. Detector data are read from the stream's `data` dataset, with a fallback to the NeXus `entry/fastcs_eiger/data` dataset. Angle and monitor paths follow the beamline format used in the supplied scans.
4. Set the fitting q range, bin count, iterations, and whether calibration-scan frame effects may be fitted. Defaults are q = 200-400 nm^-1, 800 bins, ten iterations, and frame effects enabled. `RUN_VALIDATION=True` is recommended; the default split trains on alternate positions and tests on the others, then reverses the split.
5. Run the loading/physical-check cell, then the fitting cell. Review support, rejection fractions, held-out metrics, gain maps, sensor gains, convergence history, and frame anomalies before accepting a calibration. A small training residual alone is not validation.
6. `WRITE_CALIBRATION=True` writes a new file in the final cell and reads it back to verify saved maps. Set it to False to explore without writing. Because the settings cell checks for existing outputs, rerunning after a completed export requires a new output name or a read-only inspection of the saved file.
7. Supply the resulting calibration file to `process_eiger_scan.ipynb`. Processing does not require this notebook's live kernel, raw calibration frames, or the original geometry file: geometry and physical settings are embedded in the HDF5.

## Physical Corrections

All fitting and integration use absolute solid angle, polarization, and modeled CdTe absorbed fraction. pyFAI's usual relative solid-angle factor is augmented by pixel_area / distance^2 for each scan position. The geometry check verifies predominantly vertical motion in lab Y; in this geometry horizontal degree of linear polarization 0.95 uses pyFAI factor +0.95. If '95%' means 95% horizontal intensity and 5% vertical intensity instead, the factor is 0.90. Do not guess the convention for another geometry.

CdTe energy and thickness come from metadata, with configured density 6.2 g/cm^3. xraylib total attenuation and a planar Beer-Lambert path-length model supply the normal and incidence-dependent absorbed fractions. This is not measured photon-counting quantum efficiency: escape, charge-sharing, threshold, and count-rate effects can differ. No thickness-related parallax model is fitted.

Native bad pixels, two-pixel outer edges, and documented sensor seams are excluded. Nonfinite and high/fill-sentinel counts are dynamically excluded. The exact native mask is loaded from the detector metadata, and a detector-shaped NeXus mask is also included if present.

## What Is Fitted

A free common log intensity is estimated for each q bin, alternating with pixel log gains, eight sensor log gains, and optional frame log gains relative to measured I0. Pixel means are constrained within each sensor; sensor and frame log gains have zero-mean gauges. Bounds, shrinkage, and robust weights control weakly constrained effects. Stable biased pixels can be learned as gains; statistically inconsistent contributions are rejected after warm-up. Persistent bad-pixel candidates require repeated training observations and outliers.

The fitter uses pixel-centre q-bin assignment, while integrations use pyFAI bbox/CSR splitting. The fit uses a 0.3% fractional residual floor, at least eight observations per pixel, five-sigma rejection, and pixel shrinkage weight 300. Advanced choices appear in `fit_settings`; they are model settings, not measured uncertainty. Pixel/sensor/frame decomposition is not uniquely physically identifiable from these scans, despite numerical gauges.

The cache for the default 138-frame scan is about 382 MiB, plus working arrays. Sparse scans may not constrain a pixel map. If support checks fail, first choose a suitable dense scan or supported q range; do not lower thresholds simply to make a fit succeed.

## Validation and I0 Interpretation

Test frames do not enter detector-gain estimation. Only training-selected permanent masks and raw invalid/saturation masks are used in primary validation: test pixels are not discarded because they disagree with the fitted reference. The notebook reports physical-only and detector-corrected overlap.

A separate frame-normalization diagnostic infers each test-frame gain from its lower-q portion and scores the higher-q portion after a guard. Whole-range overlap after these test-frame gains is descriptive. Frame effects may represent monitor error, illumination, acquisition timing, sample change, or residual model error; they are not an automatic diagnosis that I0 is unreliable. Fitted source-scan frame gains are stored as diagnostics and must not be transferred to another scan.

## Calibration HDF5 Layout

The schema identifier is `eiger_joint_calibration_v1`:

- `geometry_json`: complete geometry used by the fit.
- `physical/`: polarization, energy, material, thickness, density, attenuation, and transmission metadata.
- `corrections/`: pixel divisor, sensor divisor and gains, combined divisor, permanent candidate mask, source native/boundary mask, sensor IDs, observation counts, and outlier fractions.
- `fit_diagnostics/`: source angles/I0, source-only frame gains, fit indices, shared profile, supported q bins, convergence history, and held-out validation.
- Root attributes: source scan, schema/version, software version, fitting settings, and exploratory status.

Raw counts are divided by `combined_divisor`; raw-count variance is divided by its square. Never apply this combined divisor again to already corrected data. A calibration remains exploratory until it transfers to independent reference scans. Counting errors do not cover calibration, monitor, physical-model uncertainty, or neighbouring-bin covariance.

## Troubleshooting

A geometry/scan wavelength or orientation mismatch is an error, not a reason to disable the check. Missing companion files require explicit path settings. The OpenCL import warning is harmless for the selected CPU method. Empty-CSR warnings can occur for frames with no contribution in the requested integration q range; unsupported bins are not treated as measured intensity. If SciPy reports an unsupported NumPy version, correct the kernel environment before fitting.

The shared helper module contains the reusable implementation. The settings and final export cells are the only places a colleague normally needs to change; existing experimental notebooks and earlier output files are not modified.

In [ ]:
import sys
print('Python executable:', sys.executable)
print('Python version:', sys.version)

In [ ]:
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
import xraylib

REPO_DIRECTORY = Path.cwd().resolve()
while REPO_DIRECTORY != REPO_DIRECTORY.parent and not (REPO_DIRECTORY / 'eiger_workflow.py').exists():
    REPO_DIRECTORY = REPO_DIRECTORY.parent
if not (REPO_DIRECTORY / 'eiger_workflow.py').exists():
    raise FileNotFoundError('Could not find eiger_workflow.py; launch Jupyter from the repository or a subdirectory.')
DATA_DIRECTORY = REPO_DIRECTORY / 'data'
OUTPUT_DIRECTORY = REPO_DIRECTORY / 'outputs'
OUTPUT_DIRECTORY.mkdir(exist_ok=True)
SCAN_PATH = DATA_DIRECTORY / 'i15-1-99877.nxs'
GEOMETRY_PATH = DATA_DIRECTORY / 'i15-1-98779_2026-10-01_11-51-17_eiger_goniometer_calibration.json'
CALIBRATION_OUTPUT = OUTPUT_DIRECTORY / f'{SCAN_PATH.stem}_detector_corrections.h5'
POLARIZATION_FACTOR = 0.95
SENSOR_DENSITY_G_CM3 = 6.2
FIT_Q_RANGE_NM = (200.0, 400.0)
FIT_BIN_COUNT = 800
FIT_ITERATIONS = 10
FIT_FRAME_GAINS = True
RUN_VALIDATION = True
WRITE_CALIBRATION = True
DETECTOR_PATH = None
MONITOR_PATH = None
METADATA_PATH = None

assert SCAN_PATH.exists() and GEOMETRY_PATH.exists()
assert not CALIBRATION_OUTPUT.exists(), 'Choose a new CALIBRATION_OUTPUT; existing files are never overwritten.'
print('Repository:', REPO_DIRECTORY.resolve())
print('Calibration scan:', SCAN_PATH.resolve())
print('Geometry:', GEOMETRY_PATH.resolve())
print('New calibration output:', CALIBRATION_OUTPUT.resolve())
print('Imports available; editable settings initialized.')

In [ ]:
import sys
import json
import importlib

if str(REPO_DIRECTORY.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIRECTORY.resolve()))
import eiger_workflow as workflow
importlib.reload(workflow)

scan = workflow.load_scan(SCAN_PATH, DETECTOR_PATH, MONITOR_PATH, METADATA_PATH)
geometry_config = json.loads(GEOMETRY_PATH.read_text())
physical_model = workflow.PhysicalModel(geometry_config, scan, POLARIZATION_FACTOR, SENSOR_DENSITY_G_CM3)
geometry_checks = physical_model.vertical_check()
boundary_mask, sensor_ids = workflow.detector_layout(scan.shape)
base_mask = scan.native_mask | boundary_mask
sensor_ids[base_mask] = -1
workflow.constant_pattern_check(physical_model, base_mask, FIT_Q_RANGE_NM, FIT_BIN_COUNT)
fit_settings = {
    'q_range': list(FIT_Q_RANGE_NM), 'npt': FIT_BIN_COUNT, 'iterations': FIT_ITERATIONS,
    'fit_frame_gains': FIT_FRAME_GAINS, 'fractional_floor': 0.003, 'reject_z': 5.0,
    'minimum_observations': 8, 'shrink_weight': 300.0, 'frame_prior_sigma': 0.01,
}
print('Scan frames / shape:', len(scan.angles), scan.shape)
print('Angle range:', scan.angles.min(), scan.angles.max())
print('Measured I0 range:', scan.i0.min(), scan.i0.max())
print('Geometry checks:', geometry_checks)
print('Native / boundary-combined mask:', scan.native_mask.sum(), base_mask.sum())
print('CdTe normal transmission / absorbed fraction:', physical_model.transmission, physical_model.efficiency)
print('Constant-pattern physical-correction check passed.')

In [ ]:
cache = workflow.build_cache(scan, physical_model, base_mask, FIT_Q_RANGE_NM, FIT_BIN_COUNT)
contributing_indices = np.array(sorted(cache), dtype=int)
fit_options = {key: value for key, value in fit_settings.items() if key != 'q_range'}
print('Cached contributing positions:', len(contributing_indices))
print('Cached observations:', sum(len(frame['pixels']) for frame in cache.values()))
print('Cache memory (MiB):', sum(array.nbytes for frame in cache.values() for array in frame.values()) / 2**20)

validation_reports = []
validation_fits = []
if RUN_VALIDATION:
    for fold_name, train_indices, test_indices in (
        ('Even trains / odd tests', contributing_indices[contributing_indices % 2 == 0], contributing_indices[contributing_indices % 2 == 1]),
        ('Odd trains / even tests', contributing_indices[contributing_indices % 2 == 1], contributing_indices[contributing_indices % 2 == 0]),
    ):
        print('\n', fold_name)
        fold_fit = workflow.fit_response(cache, train_indices, sensor_ids, len(scan.angles), **fit_options)
        validation = workflow.validate_response(scan, physical_model, cache, fold_fit, test_indices, base_mask, FIT_Q_RANGE_NM, FIT_BIN_COUNT)
        validation_reports.append({'fold': fold_name, 'metrics': validation['summary'], 'frame_gain_checks': validation['frame_rows']})
        validation_fits.append(fold_fit)
        for row in validation['summary']:
            print(row)
    fold_sensor_difference = 100 * np.sqrt(np.mean((np.log(validation_fits[0]['sensor_gains']) - np.log(validation_fits[1]['sensor_gains']))**2))
    print('Sensor log-gain RMS difference between folds (%):', fold_sensor_difference)
else:
    print('WARNING: independent frame validation is disabled; fit diagnostics alone are not validation.')

print('\nFull calibration-scan fit')
calibration_fit = workflow.fit_response(cache, contributing_indices, sensor_ids, len(scan.angles), **fit_options)
print('Sensor gains:', calibration_fit['sensor_gains'])
print('Persistent bad-pixel candidates:', calibration_fit['bad_pixel_mask'].sum())
print('Largest calibration-scan frame gains:')
for index in sorted(contributing_indices, key=lambda index: abs(np.log(calibration_fit['frame_gains'][index])), reverse=True)[:8]:
    print(f"angle={scan.angles[index]:.3f}; measured I0={scan.i0[index]:.8f}; intensity gain={calibration_fit['frame_gains'][index]:.6f}")

fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
for plot_axis, field, title in ((axes[0, 0], 'pixel_divisor', 'Pixel-response divisor'), (axes[0, 1], 'combined_divisor', 'Pixel x sensor divisor')):
    image = plot_axis.imshow(calibration_fit[field], origin='lower', aspect='auto', cmap='coolwarm', vmin=0.95, vmax=1.05)
    plot_axis.set(title=title, xlabel='Detector column', ylabel='Detector row')
    fig.colorbar(image, ax=plot_axis)
axes[1, 0].plot(np.arange(1, 9), calibration_fit['sensor_gains'], marker='o')
axes[1, 0].set(title='Eight sensor gains', xlabel='Sensor region', ylabel='Divisor gain')
axes[1, 1].plot(scan.angles[contributing_indices], calibration_fit['frame_gains'][contributing_indices], marker='.')
axes[1, 1].axhline(1, color='black', linewidth=0.7)
axes[1, 1].set(title='Source-scan frame effects, not transferable', xlabel='Angle (degrees)', ylabel='Residual intensity gain after measured I0')
plt.show()

In [ ]:
calibration_file_written = None
if WRITE_CALIBRATION:
    calibration_file_written = workflow.save_calibration(
        CALIBRATION_OUTPUT, scan, physical_model, calibration_fit, base_mask,
        sensor_ids, fit_settings, validation_reports,
    )
    saved = workflow.load_calibration(calibration_file_written)
    np.testing.assert_array_equal(saved['combined_divisor'], calibration_fit['combined_divisor'])
    print('Created and read-back verified:', calibration_file_written)
    print('Schema:', workflow.SCHEMA)
    print('Use this file as CALIBRATION_PATH in process_eiger_scan.ipynb.')
    print('Frame gains are diagnostics for the source scan and are not transferable detector corrections.')
else:
    print('Export disabled. Inspect calibration_fit and validation_reports; set WRITE_CALIBRATION=True to save.')